# 4.5.2 The Transformer Architecture

Multi-head attention + Add&Norm + FFN encoder block — from scratch in numpy.

In [ ]:
import numpy as np

def softmax(x,ax=-1):
    x=x-x.max(axis=ax,keepdims=True); e=np.exp(x); return e/e.sum(axis=ax,keepdims=True)

def layer_norm(x,eps=1e-6):
    return (x-x.mean(-1,keepdims=True))/(x.std(-1,keepdims=True)+eps)

def mha(X,Wq,Wk,Wv,Wo,nh):
    seq,dm=X.shape; dk=dm//nh
    Q=X@Wq; K=X@Wk; V=X@Wv
    Q=Q.reshape(seq,nh,dk); K=K.reshape(seq,nh,dk); V=V.reshape(seq,nh,dk)
    heads=[softmax(Q[:,h,:]@K[:,h,:].T/np.sqrt(dk))@V[:,h,:] for h in range(nh)]
    return np.concatenate(heads,-1)@Wo

rng=np.random.default_rng(42); L,dm,nh,dff=6,16,2,32
Wq,Wk,Wv,Wo=[rng.standard_normal((dm,dm))*.1 for _ in range(4)]
X=rng.standard_normal((L,dm)); out_attn=mha(X,Wq,Wk,Wv,Wo,nh)
print('MHA output:',out_attn.shape)

In [ ]:
# Full encoder block
W1=rng.standard_normal((dm,dff))*.1; b1=np.zeros(dff); W2=rng.standard_normal((dff,dm))*.1; b2=np.zeros(dm)
X2=layer_norm(X+out_attn)
ff=np.maximum(0,X2@W1+b1)@W2+b2; X3=layer_norm(X2+ff)
print('Encoder block output:',X3.shape)
print('Mean per pos:',X3.mean(-1).round(3))  # should be ~0 due to LayerNorm